# Barotropic Decaying: Visualization Guide

Reference visualizations for the **unforced** barotropic vorticity equation at T85.
Same plot suite as `barotropic_visualization.ipynb` (which uses stochastic stirring),
but here the dynamics are deterministic: there is no `stirring_nml`, and all randomness
across sims comes from a seeded initial-condition sampler in
`sim/barotropic_decaying.py`.

Purpose: confirm that the integration produces sensible vorticity fields and to act as a
side-by-side comparison reference against the stirring-driven dataset.

Data variables used:
- `ucomp`: zonal (east-west) wind component
- `vcomp`: meridional (north-south) wind component
- `vor`: relative vorticity
- `stream`: streamfunction
- `pv`: potential vorticity

---
## Setup: Load Simulation Data

In [ ]:
SIM_DIR = "../../output/barotropic_decaying-T85/simulations/0"

In [ ]:
import json
from pathlib import Path

import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 120

from ml.data.isca_dataset import fix_time_units
from ml.diagnostics import (
    area_mean,
    mean_enstrophy,
    zonal_anomaly,
    lat_weighted_zonal_power_spectrum,
)

ds = xr.open_mfdataset(
    f"{SIM_DIR}/run*/atmos_daily.nc",
    combine="by_coords",
    data_vars="minimal",
    coords="minimal",
    compat="override",
    decode_times=False,
    preprocess=fix_time_units,
)
ds = xr.decode_cf(ds, decode_times=xr.coders.CFDatetimeCoder(use_cftime=True), decode_timedelta=True)

lon = ds.lon.values
lat = ds.lat.values

EARTH_RADIUS = 6.371e6  # m

print(f"Loaded: {dict(ds.sizes)}")
print(f"Time steps: {len(ds.time)}")
print(f"Grid: {len(lat)} x {len(lon)}")

### Sampled initial condition

Each sim under `output/barotropic_decaying-T85/simulations/<i>/` writes an `ic.json`
with the IC parameters drawn from a seed-`i` RNG by `sample_ic` in
`sim/barotropic_decaying.py`. Print it for context: the same seed always
reproduces the same IC, so this notebook is fully reproducible from the seed alone.

In [ ]:
ic_path = Path(SIM_DIR) / "ic.json"
if ic_path.exists():
    ic = json.loads(ic_path.read_text())
    print(json.dumps(ic, indent=2))
else:
    print(f"(no ic.json at {ic_path}; was this sim produced by barotropic_decaying.py?)")

---
## 1.1 Vorticity Evolution (Multi-Panel Snapshots)

Spatial distribution of relative vorticity $\zeta$ at multiple time steps.
Color = vorticity in $\text{s}^{-1}$ (red = cyclonic/positive, blue = anticyclonic/negative).
All panels share the same color scale (98th percentile robust scaling).

**What to look for:**
- A coherent wave pattern at $t=0$ matching the sampled `m_0`, `eddy_lat`, `eddy_width`
- Wave packet propagating and dispersing across panels
- Amplitude attenuation over time from hyperdiffusion (no source replenishes it)

In [ ]:
num_snaps = 10
total = len(ds.time)
indices = np.linspace(0, total - 1, num_snaps).astype(int)

vor = ds.vor.values
vlim = np.percentile(np.abs(vor), 98)

ncols = 5
nrows = int(np.ceil(num_snaps / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(18, 4 * nrows), sharey=True)
for i, idx in enumerate(indices):
    ax = axes.flat[i]
    im = ax.pcolormesh(lon, lat, vor[idx], cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
    ax.set_title(f"t={idx}")
    ax.set_xlabel("Longitude [deg]")
    if i % ncols == 0:
        ax.set_ylabel("Latitude [deg]")
for j in range(num_snaps, nrows * ncols):
    axes.flat[j].set_visible(False)
fig.colorbar(im, ax=axes, label="Vorticity [s^-1]", shrink=0.8)
fig.suptitle("Vorticity Evolution (unforced)", y=1.02)
plt.tight_layout()
plt.show()

---
## 1.2 Streamfunction

The streamfunction $\psi$ at a single snapshot. In 2D incompressible flow, streamlines
are contours of $\psi$ and the flow is tangent to them everywhere.

Black contour lines = streamlines (iso-$\psi$ curves).
Tightly packed contours = strong flow. Closed contours = vortices.

**What to look for:**
- Closed contours around vorticity centers
- Large-scale wave pattern set by the IC wavenumber `m_0`

In [ ]:
t_idx = len(ds.time) // 2
psi = ds.stream.isel(time=t_idx).values

fig, ax = plt.subplots(figsize=(12, 5))
vlim = np.percentile(np.abs(psi), 98)
im = ax.pcolormesh(lon, lat, psi, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im, ax=ax, label="Streamfunction [m^2 s^-1]")
ax.contour(lon, lat, psi, levels=15, colors="k", linewidths=0.5, alpha=0.4)
ax.set_title(f"Streamfunction (time index {t_idx})")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
plt.tight_layout()
plt.show()

---
## 1.3 Zonal Mean Zonal Wind

Zonal mean (longitude-averaged) zonal wind $[u](\phi)$ as a function of latitude,
plotted for every time step. Each line is one snapshot, colored by time.

Without stirring there is no momentum-flux source replenishing the jet. Eddy-mean-flow
interactions still redistribute momentum, but the overall envelope is expected to
decay rather than equilibrate.

**What to look for:**
- A clear jet around the IC latitude in the early lines (low time index)
- Amplitude decay across later lines
- Some latitudinal shift if the wave packet drifts

In [ ]:
u = ds.ucomp.values
ntime = u.shape[0]
u_zonal = u.mean(axis=2)

fig, ax = plt.subplots(figsize=(6, 7))
cmap = plt.cm.viridis
colors = cmap(np.linspace(0, 1, ntime))
for t in range(ntime):
    ax.plot(u_zonal[t], lat, color=colors[t], alpha=0.5, linewidth=0.8)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, ntime - 1))
sm.set_array([])
fig.colorbar(sm, ax=ax, label="Time index")
ax.axvline(0, color="k", linewidth=0.5)
ax.set_xlabel("Zonal Mean U [m/s]")
ax.set_ylabel("Latitude [deg]")
ax.set_title("Zonal Mean Zonal Wind")
plt.tight_layout()
plt.show()

---
## 1.4 Hovmoller Diagram (Meridional Wind)

Time-longitude cross section of meridional wind $v$ at $\approx 45^\circ$N.
Since the zonal mean of $v$ is near zero, any signal in $v$ is direct wave activity,
making it a cleaner indicator of Rossby wave propagation than vorticity.

Diagonal bands sloping right = eastward propagation; left = westward.
The slope gives the phase speed. Vertical bands = stationary waves.

**What to look for:**
- Clean diagonal bands from the initial wave packet
- Slope and direction set by the dispersion relation, not by forcing
- Bands fade in time as the wave dissipates

In [ ]:
target_lat = 45.0
lat_idx = np.argmin(np.abs(lat - target_lat))
v = ds.vcomp.values
v_at_lat = v[:, lat_idx, :]

time_indices = np.arange(len(ds.time))
vlim = np.percentile(np.abs(v_at_lat), 98)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.pcolormesh(lon, time_indices, v_at_lat, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im, ax=ax, label="v [m/s]")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Time index")
ax.set_title(f"Hovmoller: Meridional Wind at {lat[lat_idx]:.1f}N")
plt.tight_layout()
plt.show()

---
## 1.5 Enstrophy Time Series

Area-weighted global enstrophy $\mathcal{E} = \langle \frac{1}{2} \zeta^2 \rangle$ as a
function of time, normalized by the initial value.

Inviscid 2D dynamics conserves enstrophy exactly. With `damping_option =
resolution_dependent` and `damping_order = 4` (del^8 hyperdiffusion) and no source,
enstrophy is expected to decay monotonically as energy cascades to small scales
and is removed at the truncation wavenumber.

**What to look for:**
- Monotonic decay (modulo numerical noise)
- No spikes, no growth: any deviation from monotonic decay is a red flag
- Decay rate sets how non-trivial the operator looks to the FNO: too fast and
  the late-time fields are featureless

In [ ]:
vor = ds.vor.values
ntime = vor.shape[0]
enstrophy = mean_enstrophy(vor, lat)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(ntime), enstrophy / enstrophy[0], marker="o", markersize=3)
ax.axhline(1.0, color="k", linestyle="--", linewidth=0.8, label="Initial value")
ax.set_xlabel("Time index")
ax.set_ylabel("Enstrophy / E_0")
ax.set_title("Enstrophy Time Series (Normalized)")
ax.legend()
plt.tight_layout()
plt.show()

---
## 1.6 Rossby Wave Diagnostics

Rossby waves are planetary-scale waves that propagate along the jet due to the
gradient of potential vorticity. In the raw fields they can be masked by smaller-scale
structure. To isolate them, we spectrally filter the meridional wind $v$ in the zonal
direction, keeping only low wavenumbers.

**Filtered Hovmoller (top row):** Each panel shows a single zonal wavenumber of $v$
at the jet latitude. Diagonal bands = propagating waves.

**Planetary vs synoptic (bottom left/center):** Grouped wavenumber bands.
$k = 1$-$4$ = planetary scale, $k = 4$-$8$ = synoptic scale.

**Eddy streamfunction (bottom right):** The streamfunction with its zonal mean removed,
$\psi' = \psi - [\psi]$. The unforced run typically shows a single coherent wave train
tied to the initial `m_0`.

**What to look for:**
- The $k = m_0$ panel should dominate at early times
- Coherent diagonal bands; intensity fades as the wave dissipates
- Eddy streamfunction shows a clean wave-$m_0$ pattern, not turbulent fuzz

In [ ]:
target_lat = 45.0
lat_idx = np.argmin(np.abs(lat - target_lat))
v_at_lat = ds.vcomp.values[:, lat_idx, :]
v_hat = np.fft.rfft(v_at_lat, axis=1)
ntime_steps = len(ds.time)
time_ax = np.arange(ntime_steps)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for i, k_show in enumerate([3, 4, 5]):
    ax = axes[0, i]
    v_hat_k = np.zeros_like(v_hat)
    v_hat_k[:, k_show] = v_hat[:, k_show]
    v_k = np.fft.irfft(v_hat_k, n=v_at_lat.shape[1], axis=1)
    vlim = np.percentile(np.abs(v_k), 98)
    im = ax.pcolormesh(lon, time_ax, v_k, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
    fig.colorbar(im, ax=ax, label="v [m/s]")
    ax.set_title(f"Wavenumber k={k_show}")
    ax.set_xlabel("Longitude [deg]")
    ax.set_ylabel("Time index")

ax = axes[1, 0]
v_hat_filt = np.zeros_like(v_hat)
v_hat_filt[:, 1:5] = v_hat[:, 1:5]
v_filt = np.fft.irfft(v_hat_filt, n=v_at_lat.shape[1], axis=1)
vlim = np.percentile(np.abs(v_filt), 98)
im = ax.pcolormesh(lon, time_ax, v_filt, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im, ax=ax, label="v [m/s]")
ax.set_title("Filtered k=1-4 (planetary)")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Time index")

ax = axes[1, 1]
v_hat_filt2 = np.zeros_like(v_hat)
v_hat_filt2[:, 4:9] = v_hat[:, 4:9]
v_filt2 = np.fft.irfft(v_hat_filt2, n=v_at_lat.shape[1], axis=1)
vlim = np.percentile(np.abs(v_filt2), 98)
im = ax.pcolormesh(lon, time_ax, v_filt2, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im, ax=ax, label="v [m/s]")
ax.set_title("Filtered k=4-8 (synoptic)")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Time index")

ax = axes[1, 2]
t_mid = ntime_steps // 2
psi_snap = ds.stream.isel(time=t_mid).values
psi_eddy = zonal_anomaly(psi_snap)
vlim = np.percentile(np.abs(psi_eddy), 98)
im = ax.pcolormesh(lon, lat, psi_eddy, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im, ax=ax, label="psi' [m^2 s^-1]")
ax.set_title(f"Eddy Streamfunction (t={t_mid})")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")

fig.suptitle(f"Rossby Wave Diagnostics at {lat[lat_idx]:.1f}N", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
# Part 2: General Fluid Diagnostics

These techniques apply to any fluid system with a velocity field.

## 2.1 Wind Quiver Plot

Velocity field as arrows overlaid on a vorticity color map.
Arrow direction = wind direction, arrow length = wind speed.
Arrows plotted every 4th grid point, clipped to $\pm 70^\circ$ latitude
(the $1/\cos\phi$ factor diverges at poles, producing misleading arrow lengths).

**What to look for:**
- Arrows circulating around vorticity centers
- A clear large-scale wave pattern set by `m_0`

In [ ]:
t_idx = len(ds.time) // 2
u_snap = ds.ucomp.isel(time=t_idx).values
v_snap = ds.vcomp.isel(time=t_idx).values
vor_snap = ds.vor.isel(time=t_idx).values

vlim = np.percentile(np.abs(vor_snap), 98)
skip = 4
lat_mask = np.abs(lat) <= 70.0
lat_q = lat[lat_mask]

DEG_PER_M = 180.0 / (np.pi * EARTH_RADIUS)
SECS_PER_DAY = 86400.0
LAT_2D = np.deg2rad(lat)[:, None] * np.ones((1, len(lon)))
cos_lat_2d = np.cos(LAT_2D).clip(1e-6)
u_deg = u_snap * DEG_PER_M / cos_lat_2d * SECS_PER_DAY
v_deg = v_snap * DEG_PER_M * SECS_PER_DAY

fig, ax = plt.subplots(figsize=(12, 5))
ax.pcolormesh(lon, lat, vor_snap, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto", alpha=0.6)
ax.quiver(
    lon[::skip], lat_q[::skip],
    u_deg[lat_mask, :][::skip, ::skip],
    v_deg[lat_mask, :][::skip, ::skip],
)
ax.set_title("Wind Vectors on Vorticity (arrows: deg/day)")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")
plt.tight_layout()
plt.show()

---
## 2.2 Kinetic Energy Spectrum

Area-weighted zonal kinetic energy spectrum $E(k)$ on log-log axes.
Dashed line = $k^{-3}$ reference (expected for 2D enstrophy cascade).

**What to look for:**
- A peak at $k \approx m_0$ early in the run
- The high-$k$ tail clipped sharply by the order-4 hyperdiffusion (steeper than $k^{-3}$)
- No pile-up at the highest $k$

In [ ]:
t_idx = len(ds.time) // 2
u_snap = ds.ucomp.isel(time=t_idx).values

E_k = lat_weighted_zonal_power_spectrum(u_snap, lat)
k = np.arange(len(E_k))
k[0] = 1

fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(k[1:], E_k[1:], label="KE spectrum")
k_ref = np.array([2, 30])
ax.loglog(k_ref, E_k[2] * (k_ref / 2.0) ** (-3), "k--", label="$k^{-3}$")
ax.set_xlabel("Zonal wavenumber k")
ax.set_ylabel("Energy")
ax.set_title("Kinetic Energy Spectrum")
ax.legend()
plt.tight_layout()
plt.show()

---
## 2.3 Kinetic Energy Time Series

Area-weighted global kinetic energy $KE = \langle \frac{1}{2} u^2 \rangle$, normalized
by the initial value. Without forcing, KE is expected to decay; comparing the decay
rate against enstrophy (1.5) tells you which scales are losing energy first.

In [ ]:
u = ds.ucomp.values
ke = area_mean(0.5 * u ** 2, lat)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(len(ke)), ke / ke[0], marker="o", markersize=3)
ax.axhline(1.0, color="k", linestyle="--", linewidth=0.8)
ax.set_xlabel("Time index")
ax.set_ylabel("KE / KE_0")
ax.set_title("Kinetic Energy Time Series (Normalized)")
plt.tight_layout()
plt.show()

---
## 2.4 Jet Stream Structure (Time-Mean Zonal Wind)

Zonal wind $u$ averaged over the full integration. Time-averaging removes transient
eddies. In the unforced run the time mean is dominated by whatever zonal-mean
component is established by the IC and modified by eddy fluxes before decay.

**Left panel:** Lat-lon map of $\overline{u}$.

**Right panel:** Zonal + time mean $[\overline{u}](\phi)$ profile. Red shading = eastward,
blue = westward.

In [ ]:
u_mean = ds.ucomp.mean(dim="time").values

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
vlim = np.percentile(np.abs(u_mean), 98)
im = ax.pcolormesh(lon, lat, u_mean, cmap="RdBu_r", vmin=-vlim, vmax=vlim, shading="auto")
fig.colorbar(im, ax=ax, label="u [m/s]")
ax.set_title("Time-Mean Zonal Wind (u)")
ax.set_xlabel("Longitude [deg]")
ax.set_ylabel("Latitude [deg]")

ax = axes[1]
u_profile = u_mean.mean(axis=1)
ax.plot(u_profile, lat, "b-", linewidth=2)
ax.axvline(0, color="k", linewidth=0.5)
ax.fill_betweenx(lat, 0, u_profile, where=u_profile > 0, alpha=0.3, color="red")
ax.fill_betweenx(lat, 0, u_profile, where=u_profile < 0, alpha=0.3, color="blue")
ax.set_xlabel("U [m/s]")
ax.set_ylabel("Latitude [deg]")
ax.set_title("Zonal + Time Mean U Profile")

fig.suptitle("Jet Stream Structure", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 2.5 Wind Speed and Direction

Wind speed as background color (magma colormap) with normalized direction arrows
overlaid. All arrows have the same length; magnitude is shown only by the background
color.

Five evenly spaced snapshots show the wave packet propagating and the speed envelope
shrinking as enstrophy is dissipated.

In [ ]:
num_snaps = 5
total = len(ds.time)
indices = np.linspace(0, total - 1, num_snaps).astype(int)
subsample = 6

fig, axes = plt.subplots(1, num_snaps, figsize=(18, 4), sharey=True)

speed_all = np.sqrt(ds.ucomp.values ** 2 + ds.vcomp.values ** 2)
vmax_color = np.percentile(speed_all, 95)

for i, idx in enumerate(indices):
    ax = axes[i]
    u_snap = ds.ucomp.isel(time=idx).values
    v_snap = ds.vcomp.isel(time=idx).values
    s = np.sqrt(u_snap ** 2 + v_snap ** 2)

    im = ax.pcolormesh(lon, lat, s, cmap="magma", vmin=0, vmax=vmax_color, shading="auto", alpha=0.5)

    skip = (slice(None, None, subsample), slice(None, None, subsample))
    u_norm = u_snap[skip] / (s[skip] + 1e-10)
    v_norm = v_snap[skip] / (s[skip] + 1e-10)
    ax.quiver(
        lon[skip[1]], lat[skip[0]], u_norm, v_norm,
        color="black", scale=25, pivot="middle", width=0.005,
        headwidth=2.5, headlength=3, headaxislength=3,
    )
    ax.set_title(f"t={idx}")
    ax.set_xlabel("Longitude [deg]")
    if i == 0:
        ax.set_ylabel("Latitude [deg]")

fig.colorbar(im, ax=axes, label="Speed [m/s]", shrink=0.8)
fig.suptitle("Wind Speed and Direction (Normalized Vectors)", y=1.02)
plt.tight_layout()
plt.show()

---
# Part 3: Globe Projections (Cartopy)

Orthographic projections show the data on a sphere, giving a more intuitive sense
of spatial scale and polar geometry than flat lat-lon maps. Requires `cartopy`.

In [ ]:
import cartopy.crs as ccrs

## 3.1 Vorticity on the Globe

Four orthographic views of the vorticity field at a single snapshot:
North Pole, South Pole, and two mid-latitude views from opposite sides.

Unlike the stirring run, both hemispheres are dynamically equivalent here unless
the sampled `eddy_lat` places the IC perturbation in one of them. Expect activity
concentrated near the sampled latitude, the opposite hemisphere quiet.

In [ ]:
pc = ccrs.PlateCarree()

t_idx = len(ds.time) // 2
vor_snap = ds.vor.isel(time=t_idx).values
vlim = np.percentile(np.abs(vor_snap), 98)

views = [
    ("North Pole", ccrs.Orthographic(0, 90)),
    ("South Pole", ccrs.Orthographic(0, -90)),
    ("Atlantic", ccrs.Orthographic(-30, 30)),
    ("Pacific", ccrs.Orthographic(180, 30)),
]

fig = plt.figure(figsize=(20, 5))
ax_list = []
for i, (title, proj) in enumerate(views):
    ax = fig.add_subplot(1, 4, i + 1, projection=proj)
    im = ax.pcolormesh(lon, lat, vor_snap, cmap="RdBu_r", vmin=-vlim, vmax=vlim,
                       shading="auto", transform=pc)
    ax.set_global()
    ax.gridlines(alpha=0.3)
    ax.set_title(title)
    ax_list.append(ax)

fig.colorbar(im, ax=ax_list, label="Vorticity [s^-1]", shrink=0.8,
             orientation="horizontal", pad=0.05)
fig.suptitle(f"Vorticity on the Globe (t={t_idx})", fontsize=14, fontweight="bold")
plt.show()

---
## 3.2 Multi-Field Globe Comparison

Three fields on the same globe view side by side: wind speed, time-mean zonal wind,
and streamfunction. Comparing them on the same projection makes it easy to see how
the (now decaying) eddies and large-scale circulation relate spatially.

In [ ]:
t_idx = len(ds.time) // 2
proj = ccrs.Orthographic(-30, 30)

fields = [
    ("Wind Speed", np.sqrt(ds.ucomp.isel(time=t_idx).values ** 2 +
                           ds.vcomp.isel(time=t_idx).values ** 2),
     "magma", None, "m/s"),
    ("Time-Mean Zonal Wind", ds.ucomp.mean(dim="time").values,
     "RdBu_r", "symmetric", "m/s"),
    ("Streamfunction", ds.stream.isel(time=t_idx).values,
     "RdBu_r", "symmetric", "m^2 s^-1"),
]

fig = plt.figure(figsize=(18, 5))
for i, (title, data, cmap, mode, units) in enumerate(fields):
    ax = fig.add_subplot(1, 3, i + 1, projection=proj)
    if mode == "symmetric":
        vlim = np.percentile(np.abs(data), 98)
        kwargs = {"vmin": -vlim, "vmax": vlim}
    else:
        kwargs = {"vmin": 0, "vmax": np.percentile(data, 95)}
    im = ax.pcolormesh(lon, lat, data, cmap=cmap, shading="auto", transform=pc, **kwargs)
    ax.set_global()
    ax.gridlines(alpha=0.3)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=units, shrink=0.7)

fig.suptitle("Globe Projections", fontsize=14, fontweight="bold")
plt.show()

---
## 3.3 Vorticity Evolution on the Globe

Same as 1.1 but on an orthographic projection. Six snapshots show how the initial
wave packet propagates and decays on the sphere.

In [ ]:
num_snaps = 6
total = len(ds.time)
indices = np.linspace(0, total - 1, num_snaps).astype(int)

vor_all = ds.vor.values
vlim = np.percentile(np.abs(vor_all), 98)
proj = ccrs.Orthographic(-30, 45)

fig = plt.figure(figsize=(18, 6))
ax_list = []
for i, idx in enumerate(indices):
    ax = fig.add_subplot(1, num_snaps, i + 1, projection=proj)
    im = ax.pcolormesh(lon, lat, vor_all[idx], cmap="RdBu_r", vmin=-vlim, vmax=vlim,
                       shading="auto", transform=pc)
    ax.set_global()
    ax.gridlines(alpha=0.3)
    ax.set_title(f"t={idx}")
    ax_list.append(ax)

fig.colorbar(im, ax=ax_list, label="Vorticity [s^-1]", shrink=0.6,
             orientation="horizontal", pad=0.05)
fig.suptitle("Vorticity Evolution on the Globe", fontsize=14, fontweight="bold")
plt.show()